# Aula 12 - Cluster Analysis

Nesta prática vamos usar um exemplo concreto: uma empresa de roupas precisa transformar medidas contínuas de corpos em grupos de tamanhos. A ideia não é descobrir uma verdade absoluta, mas construir agrupamentos úteis para tomar decisões.

Você já viu KNN, então a intuição de distância já existe. Vamos começar pelo método hierárquico, depois avançar para K-Means e fechar com DBSCAN.

## 1. Preparação

Execute as células em ordem. O notebook usa dados sintéticos porque isso permite controlar grupos, sobreposição, outliers e casos ambíguos.

In [ ]:
# Pacotes necessários para executar este notebook:
# uv pip install numpy pandas plotly scipy scikit-learn
#
# Nomes que podem confundir:
# - sklearn é instalado como scikit-learn
# - pathlib e json fazem parte da biblioteca padrao do Python

from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.utils import PlotlyJSONEncoder
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

DRACULA = {
    'bg': '#282a36',
    'fg': '#f8f8f2',
    'cyan': '#8be9fd',
    'green': '#50fa7b',
    'pink': '#ff79c6',
    'red': '#ff5555',
    'yellow': '#f1fa8c',
    'purple': '#bd93f9',
    'muted': '#6272a4',
}

def apply_dracula(fig, title=None):
    fig.update_layout(
        title=title,
        paper_bgcolor=DRACULA['bg'],
        plot_bgcolor=DRACULA['bg'],
        font=dict(color=DRACULA['fg']),
        margin=dict(l=60, r=30, t=60, b=55),
    )
    fig.update_xaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    fig.update_yaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    return fig

## 2. Dataset sintético: cintura x entreperna

Vamos gerar quatro grupos naturais:

- baixo/estreito
- alto/estreito
- baixo/largo
- alto/largo

Também vamos adicionar alguns outliers. Na vida real, eles podem representar erros de medição ou grupos minoritários que a empresa ainda não atende bem.

In [ ]:
def make_body_dataset(n_per_group=50):
    specs = [
        ('baixo_estreito', 72, 72),
        ('alto_estreito', 74, 92),
        ('baixo_largo', 98, 74),
        ('alto_largo', 102, 94),
    ]
    frames = []
    for name, waist_mean, inseam_mean in specs:
        waist = rng.normal(waist_mean, 5, n_per_group)
        inseam = rng.normal(inseam_mean, 4, n_per_group)
        frames.append(pd.DataFrame({
            'cintura_cm': waist,
            'entreperna_cm': inseam,
            'grupo_gerador': name,
        }))
    outliers = pd.DataFrame({
        'cintura_cm': [88, 116, 62, 111],
        'entreperna_cm': [84, 104, 100, 66],
        'grupo_gerador': ['outlier'] * 4,
    })
    return pd.concat(frames + [outliers], ignore_index=True)

df = make_body_dataset()
features = ['cintura_cm', 'entreperna_cm']
df.head()

In [ ]:
fig_initial = px.scatter(
    df, x='cintura_cm', y='entreperna_cm',
    color_discrete_sequence=[DRACULA['cyan']],
    labels={'cintura_cm': 'Cintura (cm)', 'entreperna_cm': 'Entreperna (cm)'},
)
fig_initial.update_traces(marker=dict(size=9, line=dict(color=DRACULA['bg'], width=1)))
apply_dracula(fig_initial, 'Cintura x entreperna: dados sem rótulo')
fig_initial.show()

### Atividade 1

Antes de rodar qualquer algoritmo, responda:

1. Quantos grupos você enxerga visualmente?
2. Quais regiões parecem ambíguas?
3. O que seria um tamanho de calça justo para pessoas próximas das fronteiras?

## 3. Clustering hierárquico

O método hierárquico começa com cada ponto sozinho e vai unindo os pontos ou grupos mais próximos. O dendrograma mostra a ordem dessas fusões.

In [ ]:
X = df[features].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Z = linkage(X_scaled, method='ward')
hier_labels = fcluster(Z, t=4, criterion='maxclust')
df['cluster_hierarquico'] = hier_labels.astype(str)

fig_hier = px.scatter(
    df, x='cintura_cm', y='entreperna_cm', color='cluster_hierarquico',
    color_discrete_sequence=[DRACULA['cyan'], DRACULA['pink'], DRACULA['green'], DRACULA['yellow'], DRACULA['red']],
    labels={'cintura_cm': 'Cintura (cm)', 'entreperna_cm': 'Entreperna (cm)'},
)
fig_hier.update_traces(marker=dict(size=9))
apply_dracula(fig_hier, 'Clustering hierárquico com 4 grupos')
fig_hier.show()

In [ ]:
dendro = dendrogram(Z, no_plot=True, truncate_mode='lastp', p=20)
fig_dendro = go.Figure()
for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    fig_dendro.add_trace(go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=DRACULA['cyan'], width=3), showlegend=False))
fig_dendro.add_hline(y=np.percentile(Z[:, 2], 80), line_dash='dash', line_color=DRACULA['pink'])
fig_dendro.update_xaxes(title='Grupos/pontos', showticklabels=False)
fig_dendro.update_yaxes(title='Distância da fusão')
apply_dracula(fig_dendro, 'Dendrograma truncado')
fig_dendro.show()

### Atividade 2

Troque `method='ward'` por `complete`, `average` e `single`. O que muda?

Escreva uma frase explicando qual linkage você escolheria para apresentar a um gerente de produto.

## 4. K-Means passo a passo

K-Means representa cada cluster por um centroide. O algoritmo alterna entre atribuir pontos ao centroide mais próximo e mover cada centroide para a média dos pontos atribuídos.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=20, random_state=RANDOM_STATE)
df['cluster_kmeans'] = kmeans.fit_predict(X_scaled).astype(str)
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)

fig_kmeans = px.scatter(
    df, x='cintura_cm', y='entreperna_cm', color='cluster_kmeans',
    color_discrete_sequence=[DRACULA['cyan'], DRACULA['pink'], DRACULA['green'], DRACULA['yellow']],
    labels={'cintura_cm': 'Cintura (cm)', 'entreperna_cm': 'Entreperna (cm)'},
)
fig_kmeans.add_trace(go.Scatter(
    x=centroids_original[:, 0], y=centroids_original[:, 1],
    mode='markers', name='centroides',
    marker=dict(color=DRACULA['red'], size=18, symbol='x', line=dict(color=DRACULA['fg'], width=2)),
))
apply_dracula(fig_kmeans, 'K-Means com k = 4')
fig_kmeans.show()

In [ ]:
inertias = []
for k in range(1, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

fig_elbow = go.Figure(go.Scatter(x=list(range(1, 9)), y=inertias, mode='lines+markers', line=dict(color=DRACULA['green'], width=4)))
fig_elbow.update_xaxes(title='Número de clusters (k)', dtick=1)
fig_elbow.update_yaxes(title='Inércia / SSE')
apply_dracula(fig_elbow, 'Método do cotovelo')
fig_elbow.show()

### Atividade 3

1. Rode K-Means com `k=3`, `k=4`, `k=5` e `k=6`.
2. Compare a interpretação dos centroides.
3. Qual valor de k você defenderia se cada tamanho novo aumenta custo de produção?

## 5. DBSCAN

DBSCAN encontra regiões densas e marca pontos isolados como ruído. Para visualizar melhor a força do método, vamos usar um dataset com forma curva e alguns pontos de ruído.

In [ ]:
moons_X, _ = make_moons(n_samples=240, noise=0.07, random_state=RANDOM_STATE)
noise = rng.uniform(low=[-1.5, -0.9], high=[2.5, 1.4], size=(18, 2))
db_X = np.vstack([moons_X, noise])
dbscan = DBSCAN(eps=0.22, min_samples=6)
db_labels = dbscan.fit_predict(db_X).astype(str)
db_df = pd.DataFrame(db_X, columns=['x', 'y'])
db_df['cluster_dbscan'] = np.where(db_labels == '-1', 'ruído', db_labels)

fig_dbscan = px.scatter(
    db_df, x='x', y='y', color='cluster_dbscan',
    color_discrete_sequence=[DRACULA['cyan'], DRACULA['green'], DRACULA['red'], DRACULA['pink']],
)
fig_dbscan.update_traces(marker=dict(size=8))
apply_dracula(fig_dbscan, 'DBSCAN em dados não esféricos')
fig_dbscan.show()

### Atividade 4

Altere `eps` para `0.12`, `0.18`, `0.30` e `0.45`. Depois altere `min_samples` para `3`, `6` e `12`.

Responda:

1. Quando o algoritmo cria ruído demais?
2. Quando ele junta grupos que deveriam ficar separados?
3. O que é mais difícil de explicar para alguém fora da área: `k` ou `eps`?

## 6. Exportar Plotly para os slides

A função abaixo exporta figuras Plotly para um arquivo JavaScript com constantes. O deck HTML pode carregar esse arquivo e usar os gráficos com `<plotly-figure>`.

In [ ]:
def clean_none(obj):
    if isinstance(obj, dict):
        return {k: clean_none(v) for k, v in obj.items() if v is not None}
    if isinstance(obj, list):
        return [clean_none(v) for v in obj]
    return obj

def slide_export_path():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks' and cwd.parent.name == 'ciencia-dados':
        return cwd.parent / 'images' / 'plotly' / 'aula12_cluster_analysis_figures_from_notebook.js'
    return cwd / 'ciencia-dados' / 'images' / 'plotly' / 'aula12_cluster_analysis_figures_from_notebook.js'

def write_plotly_js(figures, output_path):
    lines = []
    for name, fig in figures.items():
        payload = clean_none(fig.to_dict())
        traces = json.dumps(payload['data'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        layout = json.dumps(payload['layout'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        lines.append(f"const {name}_TRACES = {traces};\n")
        lines.append(f"const {name}_LAYOUT = {layout};\n")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines), encoding='utf-8')

figures = {
    'AULA12_SCATTER_INITIAL': fig_initial,
    'AULA12_HIERARCHICAL_CLUSTERS': fig_hier,
    'AULA12_DENDROGRAM': fig_dendro,
    'AULA12_ELBOW': fig_elbow,
    'AULA12_DBSCAN': fig_dbscan,
}

write_plotly_js(figures, slide_export_path())

## Entrega

Escreva uma resposta curta com:

1. Qual método você escolheria para criar uma tabela inicial de tamanhos?
2. Qual método você usaria para investigar outliers ou grupos minoritários?
3. Quais limitações ou riscos sociais aparecem quando uma empresa padroniza corpos em poucos tamanhos?